# Round 2 · Conversation shape, not conversation text

**The attacker in the data:** a crew that bought aged accounts, uses unique photos and has a language
model draft and translate every message, so no two read alike. Round 1 sees none of it.

What the model does not change is the operation around it: people working shifts, many conversations
at once, and the move to another app before money is mentioned.

**Your job:** add to your Round 1 flags in `out/suspects_r2.csv` (the file is cumulative).

In [ ]:
import sys
sys.path.insert(0, "../tools")
import pandas as pd
import trailmates as tm
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

feats = pd.DataFrame.from_dict(tm.member_features(), orient="index")
round1 = tm.read_suspects("../out/suspects_r1.csv")
print(len(round1), "carried over from Round 1")
feats[["outbound_first", "inbound_first", "hour_concentration", "weekend_share",
       "early_offplatform_rate", "translated_share"]].describe().T

| feature | meaning |
|---|---|
| `outbound_first` / `inbound_first` | conversations this account opened / that others opened with it |
| `hour_concentration` | share of its messages inside its busiest ten contiguous UTC hours |
| `weekend_share` | share of its messages on Saturday or Sunday (UTC) |
| `early_offplatform_rate` | share of the conversations it opened that it asked to move to another app within 7 days |
| `translated_share` | share of its messages sent through the built-in translator |

In [ ]:
def round2(f):
    # TODO (Round 2): the starter flags accounts that open many conversations and move them off-platform
    # early. Who else on a hobby community does exactly that? Find what they do NOT share with a crew.
    return (f["outbound_first"] >= 10) & (f["early_offplatform_rate"] >= 0.15)


new = feats[round2(feats) & ~feats.index.isin(round1)]
scores = dict(round1)
scores.update({m: 1.0 + r.early_offplatform_rate for m, r in new.iterrows()})
tm.write_suspects("../out/suspects_r2.csv", scores)
print(len(new), "added;", len(scores), "in total")

In [ ]:
!{sys.executable} ../tools/checkpoint.py --round 2

## Read the scoreboard

- Look at the hours of the accounts you added: plot one account's messages per UTC hour.
- An account that keeps office hours and messages hundreds of people may be the friendliest person on
  the community. What does it never do?
- Try `translated_share` as a signal, score it, and read which cohort pays. Then never use it again.

In [ ]:
dms = tm.frames()["dms"]
sample = new.sort_values("outbound_first", ascending=False).index[:4]
for m in sample:
    hours = dms.loc[dms["sender"] == m, "ts"].dt.hour.value_counts().reindex(range(24), fill_value=0)
    hours.plot.bar(figsize=(10, 2), title=m)
    import matplotlib.pyplot as plt
    plt.show()

PASS? Go to `03_interventions.ipynb`. Stuck? `python tools/checkpoint.py --round 2 --catch-up`.